# Notebook 2: Load & Predict

This notebook represents the inference stage of the customer churn prediction workflow.

The final trained machine learning Pipeline created in Notebook 1 is loaded from its serialized artifact and used to generate predictions for new customer records.

The Pipeline contains the complete feature-engineering, preprocessing, and model steps required for inference. No model retraining or manual recreation of preprocessing is performed in this notebook.

The objective is to demonstrate that the saved Pipeline can accept raw customer input and generate churn predictions and prediction probabilities consistently after being loaded.

## Notebook Contents

1. Task 1: Load the Saved Pipeline
2. Task 2: Prepare New/Raw Data
3. Task 3: Generate Predictions
4. Task 4: Explain the Pipeline Design

## Task 1: Load the Saved Model

The final trained Pipeline created in Notebook 1 is loaded from the saved joblib artifact.

The saved artifact contains the complete feature-engineering, preprocessing, and model pipeline required for inference.

In [ ]:
import joblib
import os

pipeline_path = "model/final_pipeline.joblib"

print("Pipeline exists:", os.path.exists(pipeline_path))

loaded_pipeline = joblib.load(pipeline_path)

print("Saved Pipeline loaded successfully.")

## Task 2: Prepare New/Raw Data

A small set of unseen customer records is loaded to simulate new customer inputs.

The raw input columns expected by the saved Pipeline are used directly. No feature engineering or preprocessing is manually performed in this notebook.

## 2.1 Load Raw Customer Data

The original customer churn dataset is loaded to provide raw customer records for inference.

Only the original customer attributes are used as inputs to the saved Pipeline.

In [ ]:
import pandas as pd

df_raw = pd.read_excel(
    "dataset/Telco_customer_churn.xlsx"
)

print("Raw dataset shape:", df_raw.shape)

## 2.2 Select Unseen Customer Records

A small sample of customer records is selected to simulate new customer data that becomes available after model deployment.

A fixed random seed is used so that the inference demonstration remains reproducible.

In [ ]:
raw_customers = df_raw.sample(
    n=5,
    random_state=42
).copy()

raw_customers

## 2.3 Prepare Raw Input for Prediction

The target and target-related columns are removed because they would not be available when making predictions for new customers.

The remaining columns represent the raw customer information provided to the saved Pipeline.

In [ ]:
raw_features = raw_customers.drop(
    columns=[
        "Churn Value",
        "Churn Label",
        "Churn Score",
        "Churn Reason"
    ],
    errors="ignore"
)

raw_features

## Task 3: Generate Predictions

The loaded Pipeline is used to generate churn predictions for the selected raw customer records.

The model produces both the predicted churn class and the corresponding probability of churn for each customer. The predictions are generated directly from the raw input records using the saved Pipeline.

## 3.1 Predicted Churn Class

The saved Pipeline generates a binary churn prediction for each customer, where 0 represents no churn and 1 represents churn.

In [ ]:
predicted_class = loaded_pipeline.predict(
    raw_features
)

predicted_class

## 3.2 Prediction Probability

The model also generates a churn probability for each customer. This probability indicates the estimated likelihood of churn for the given customer record.

In [ ]:
predicted_probability = loaded_pipeline.predict_proba(
    raw_features
)[:, 1]

predicted_probability

## 3.3 Prediction Results

The customer information and model outputs are combined into a single table to make the inference results easier to interpret.

In [ ]:
prediction_results = raw_features[
    [
        "CustomerID",
        "Contract",
        "Tenure Months",
        "Monthly Charges"
    ]
].copy()

prediction_results["Predicted Churn"] = predicted_class

prediction_results["Churn Probability"] = predicted_probability

prediction_results["Predicted Churn Label"] = (
    prediction_results["Predicted Churn"]
    .map({
        0: "No Churn",
        1: "Churn"
    })
)

prediction_results

### Task 3 Summary

The saved Pipeline successfully generated both churn predictions and churn probabilities for the selected raw customer records.

The results were combined with key customer attributes into a clear prediction table, demonstrating that the loaded Pipeline can directly transform raw inputs and produce interpretable inference results.

## Task 4: Explain the Pipeline Design

The complete Pipeline combines feature engineering, preprocessing, and the trained model into a single inference workflow.

This design ensures that new customer records are processed using the same transformations that were applied during model development.

Saving the complete Pipeline is safer and more reproducible than saving only the trained model because it keeps feature engineering and preprocessing together with the model.

This ensures that the same transformations are applied to new customer data during inference and avoids the risk of inconsistent preprocessing between training and prediction.

The successful predictions generated in Task 3 demonstrate that the saved Pipeline can be loaded and applied to raw customer records without retraining or manually recreating the preprocessing steps.